# MFLS + QDA-Magnified Ensemble Benchmark
**Goal:** Stack MFLS Quadratic scoring with QDA-Magnified to improve production accuracy.

Each cell is independent — if one fails, fix it and re-run just that cell.

### Pipeline Architecture
```
Raw X → RepStack (20D) ─┬─ Magnifier → QDA (per-dim view)  ─┐
                        └─ Tensor → MFLS (per-law view)     ├─ Ensemble → Predict
                                                             ┘
```
QDA sees **fine-grained dimension detail** (20D magnified).  
MFLS sees **cross-law interactions** (5 law magnitudes, raw).  
Stacking them = a sample must look anomalous from **both perspectives** → slashes false positives.

In [1]:
# Cell 1: Imports & Setup — run this first
import sys, warnings, time, pickle
warnings.filterwarnings('ignore')
sys.path.insert(0, r'c:\amttp')

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

from udl.pipeline import UDLPipeline
from udl.datasets import load_mammography, load_shuttle

# Helper function — reusable across all cells
def evaluate(label, p, X_te, y_te):
    """Evaluate a fitted pipeline and print metrics."""
    preds = p.predict(X_te)
    probs = p.predict_proba(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
    n_pos = int((y_te == 1).sum())
    n_neg = int((y_te == 0).sum())
    f1 = f1_score(y_te, preds)
    auc = roc_auc_score(y_te, probs)
    prec = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
    fnr = fn / n_pos * 100 if n_pos else 0
    fpr = fp / n_neg * 100 if n_neg else 0
    acc = (tp + tn) / len(y_te) * 100
    
    print(f"  {label}")
    print(f"    TP={tp:>3d}  TN={tn:>4d}  FP={fp:>3d}  FN={fn:>2d}")
    print(f"    FN rate (missed):      {fnr:.2f}%  ({fn}/{n_pos})")
    print(f"    FP rate (false alarm):  {fpr:.2f}%  ({fp}/{n_neg})")
    print(f"    Precision: {prec:.1f}%   F1: {f1:.4f}   Accuracy: {acc:.1f}%   AUC: {auc:.4f}")
    
    return {"label": label, "tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "f1": f1, "auc": auc, "prec": prec, "fnr": fnr, "fpr": fpr, "acc": acc}

# Store all results for final comparison
results = {}
print("Setup complete ✓")

Setup complete ✓


## Step 2: Load Mammography Dataset
Split into train/test. This is the hardest real-world dataset (2.3% anomaly rate, 11K samples).

In [2]:
# Cell 2: Load mammography
X_mammo, y_mammo = load_mammography()
X_tr, X_te, y_tr, y_te = train_test_split(
    X_mammo, y_mammo, test_size=0.3, random_state=42, stratify=y_mammo
)
n_pos = int((y_te == 1).sum())
n_neg = int((y_te == 0).sum())
print(f"Mammography loaded: {len(X_mammo)} total, {len(X_te)} test")
print(f"  Normal: {n_neg}  |  Anomaly: {n_pos}  |  Rate: {n_pos/len(y_te)*100:.1f}%")

# Save checkpoint so we never need to reload
checkpoint = {"X_tr": X_tr, "X_te": X_te, "y_tr": y_tr, "y_te": y_te}
print("Checkpoint saved ✓")

Mammography loaded: 11183 total, 3355 test
  Normal: 3277  |  Anomaly: 78  |  Rate: 2.3%
Checkpoint saved ✓


## Step 3: Baseline — QDA-Magnified Alone
No MFLS, no ensemble. Just QDA with auto-calibrated threshold. This is our baseline to beat.

In [3]:
# Cell 3: Baseline QDA-Magnified (cost=1 and cost=10)
t0 = time.time()

# cost=1 (F1-optimal threshold)
p_base1 = UDLPipeline(projection_method='qda-magnified', cost_ratio=1.0)
p_base1.magnifier_.verbose = False
p_base1.fit(X_tr, y_tr)
r1 = evaluate("QDA-Mag baseline (cost=1, F1-optimal)", p_base1, X_te, y_te)
results["qda_cost1"] = r1

print()

# cost=10 (high-stakes)
p_base10 = UDLPipeline(projection_method='qda-magnified', cost_ratio=10.0)
p_base10.magnifier_.verbose = False
p_base10.fit(X_tr, y_tr)
r10 = evaluate("QDA-Mag baseline (cost=10, high-stakes)", p_base10, X_te, y_te)
results["qda_cost10"] = r10

print(f"\n  Elapsed: {time.time()-t0:.1f}s")

  QDA-Mag baseline (cost=1, F1-optimal)
    TP= 44  TN=3232  FP= 45  FN=34
    FN rate (missed):      43.59%  (34/78)
    FP rate (false alarm):  1.37%  (45/3277)
    Precision: 49.4%   F1: 0.5269   Accuracy: 97.6%   AUC: 0.9425

  QDA-Mag baseline (cost=10, high-stakes)
    TP= 65  TN=3120  FP=157  FN=13
    FN rate (missed):      16.67%  (13/78)
    FP rate (false alarm):  4.79%  (157/3277)
    Precision: 29.3%   F1: 0.4333   Accuracy: 94.9%   AUC: 0.9425

  Elapsed: 141.8s


## Step 4: MFLS Quadratic + QDA Ensemble (Stack)
The **stack** ensemble trains a logistic meta-classifier on `[QDA_prob, MFLS_score]`.  
MFLS Quadratic fits a degree-2 polynomial surface on law-domain magnitudes (5 laws → 20 quadratic features).

In [4]:
# Cell 4: MFLS Quadratic + Stack ensemble
t0 = time.time()

p = UDLPipeline(
    projection_method='qda-magnified',
    mfls_method='quadratic',
    ensemble='stack',
    cost_ratio=1.0
)
p.magnifier_.verbose = False
p.fit(X_tr, y_tr)

print(f"  Ensemble threshold: {p._ensemble_threshold:.4f}")
r = evaluate("MFLS-quadratic + stack (cost=1)", p, X_te, y_te)
results["quad_stack_c1"] = r

print(f"\n  Elapsed: {time.time()-t0:.1f}s")

  Ensemble threshold: 0.3206
  MFLS-quadratic + stack (cost=1)
    TP= 44  TN=3231  FP= 46  FN=34
    FN rate (missed):      43.59%  (34/78)
    FP rate (false alarm):  1.40%  (46/3277)
    Precision: 48.9%   F1: 0.5238   Accuracy: 97.6%   AUC: 0.9399

  Elapsed: 217.2s


## Step 5: MFLS Quadratic Smooth + Stack
Same as above but with tanh saturation on MFLS scores to cap runaway false alarms.

In [5]:
# Cell 5: MFLS Quadratic Smooth + Stack ensemble
t0 = time.time()

p = UDLPipeline(
    projection_method='qda-magnified',
    mfls_method='quadratic_smooth',
    ensemble='stack',
    cost_ratio=1.0
)
p.magnifier_.verbose = False
p.fit(X_tr, y_tr)

print(f"  Ensemble threshold: {p._ensemble_threshold:.4f}")
r = evaluate("MFLS-quad_smooth + stack (cost=1)", p, X_te, y_te)
results["qsmooth_stack_c1"] = r

print(f"\n  Elapsed: {time.time()-t0:.1f}s")

  Ensemble threshold: 0.3355
  MFLS-quad_smooth + stack (cost=1)
    TP= 44  TN=3228  FP= 49  FN=34
    FN rate (missed):      43.59%  (34/78)
    FP rate (false alarm):  1.50%  (49/3277)
    Precision: 47.3%   F1: 0.5146   Accuracy: 97.5%   AUC: 0.9393

  Elapsed: 248.1s


## Step 6: MFLS Logistic + Stack
Logistic regression on per-law magnitudes — simpler than quadratic but sometimes more robust.

In [6]:
# Cell 6: MFLS Logistic + Stack ensemble
t0 = time.time()

p = UDLPipeline(
    projection_method='qda-magnified',
    mfls_method='logistic',
    ensemble='stack',
    cost_ratio=1.0
)
p.magnifier_.verbose = False
p.fit(X_tr, y_tr)

print(f"  Ensemble threshold: {p._ensemble_threshold:.4f}")
r = evaluate("MFLS-logistic + stack (cost=1)", p, X_te, y_te)
results["logistic_stack_c1"] = r

print(f"\n  Elapsed: {time.time()-t0:.1f}s")

  Ensemble threshold: 0.2680
  MFLS-logistic + stack (cost=1)
    TP= 38  TN=3219  FP= 58  FN=40
    FN rate (missed):      51.28%  (40/78)
    FP rate (false alarm):  1.77%  (58/3277)
    Precision: 39.6%   F1: 0.4368   Accuracy: 97.1%   AUC: 0.9314

  Elapsed: 245.0s


## Step 7: Alternative Ensemble Modes — Product & Max-F1
- **product**: Geometric mean √(QDA × MFLS) — AND-logic, both must agree
- **max_f1**: Sweep α in `α·QDA + (1-α)·MFLS` to find best F1

In [7]:
# Cell 7: Product and Max-F1 ensemble modes (using quadratic MFLS)
t0 = time.time()

for ens_mode in ['product', 'max_f1']:
    p = UDLPipeline(
        projection_method='qda-magnified',
        mfls_method='quadratic',
        ensemble=ens_mode,
        cost_ratio=1.0
    )
    p.magnifier_.verbose = False
    p.fit(X_tr, y_tr)
    
    label = f"MFLS-quadratic + {ens_mode} (cost=1)"
    if ens_mode == 'max_f1':
        print(f"  Best alpha: {p._ensemble_alpha:.2f}")
    print(f"  Threshold: {p._ensemble_threshold:.4f}")
    r = evaluate(label, p, X_te, y_te)
    results[f"quad_{ens_mode}_c1"] = r
    print()

print(f"Elapsed: {time.time()-t0:.1f}s")

  Threshold: 0.8647
  MFLS-quadratic + product (cost=1)
    TP= 44  TN=3232  FP= 45  FN=34
    FN rate (missed):      43.59%  (34/78)
    FP rate (false alarm):  1.37%  (45/3277)
    Precision: 49.4%   F1: 0.5269   Accuracy: 97.6%   AUC: 0.9425

  Best alpha: 0.96
  Threshold: 0.9899
  MFLS-quadratic + max_f1 (cost=1)
    TP= 39  TN=3244  FP= 33  FN=39
    FN rate (missed):      50.00%  (39/78)
    FP rate (false alarm):  1.01%  (33/3277)
    Precision: 54.2%   F1: 0.5200   Accuracy: 97.9%   AUC: 0.9433

Elapsed: 445.8s


## Step 8: Cost-Sensitive Ensemble (High-Stakes)
Run the best MFLS methods with `cost_ratio=10` — penalise missed anomalies 10× more than false alarms.

In [8]:
# Cell 8: Cost-sensitive ensemble (cost=10)
t0 = time.time()

for mfls in ['quadratic', 'quadratic_smooth', 'logistic']:
    for ens in ['stack', 'max_f1']:
        p = UDLPipeline(
            projection_method='qda-magnified',
            mfls_method=mfls,
            ensemble=ens,
            cost_ratio=10.0
        )
        p.magnifier_.verbose = False
        p.fit(X_tr, y_tr)
        
        label = f"{mfls} + {ens} (cost=10)"
        r = evaluate(label, p, X_te, y_te)
        results[f"{mfls}_{ens}_c10"] = r
        print()

print(f"Elapsed: {time.time()-t0:.1f}s")

  quadratic + stack (cost=10)
    TP= 62  TN=3153  FP=124  FN=16
    FN rate (missed):      20.51%  (16/78)
    FP rate (false alarm):  3.78%  (124/3277)
    Precision: 33.3%   F1: 0.4697   Accuracy: 95.8%   AUC: 0.9399

  quadratic + max_f1 (cost=10)
    TP= 39  TN=3244  FP= 33  FN=39
    FN rate (missed):      50.00%  (39/78)
    FP rate (false alarm):  1.01%  (33/3277)
    Precision: 54.2%   F1: 0.5200   Accuracy: 97.9%   AUC: 0.9433

  quadratic_smooth + stack (cost=10)
    TP= 64  TN=3139  FP=138  FN=14
    FN rate (missed):      17.95%  (14/78)
    FP rate (false alarm):  4.21%  (138/3277)
    Precision: 31.7%   F1: 0.4571   Accuracy: 95.5%   AUC: 0.9393

  quadratic_smooth + max_f1 (cost=10)
    TP= 39  TN=3247  FP= 30  FN=39
    FN rate (missed):      50.00%  (39/78)
    FP rate (false alarm):  0.92%  (30/3277)
    Precision: 56.5%   F1: 0.5306   Accuracy: 97.9%   AUC: 0.9433

  logistic + stack (cost=10)
    TP= 65  TN=3116  FP=161  FN=13
    FN rate (missed):      16.67%  (13

## Step 9: Tiered Output — CLEAR / REVIEW / ALERT
Production deployment pattern. Pick the best ensemble and show how many cases get auto-decided vs need human review.

In [9]:
# Cell 9: Tiered output with best ensemble config
# Try best MFLS method from results above
best_mfls = 'quadratic'  # change this if another method won in Step 8
best_ens = 'stack'

p_tiered = UDLPipeline(
    projection_method='qda-magnified',
    mfls_method=best_mfls,
    ensemble=best_ens,
    cost_ratio=10.0
)
p_tiered.magnifier_.verbose = False
p_tiered.fit(X_tr, y_tr)

tiers, probs = p_tiered.predict_tiered(X_te)
n_pos = int((y_te == 1).sum())

print("  TIERED OUTPUT (Mammography, cost=10)")
print(f"  {'='*60}")
for tier_val, tier_label in [(0, "CLEAR"), (1, "REVIEW"), (2, "ALERT")]:
    mask = tiers == tier_val
    n = mask.sum()
    anom = (y_te[mask] == 1).sum()
    norm = (y_te[mask] == 0).sum()
    pct = n / len(y_te) * 100
    
    if tier_val == 0:
        leak = anom / max(n_pos, 1) * 100
        print(f"    {tier_label:>6s}: {n:>5d} ({pct:>5.1f}%)  |  {norm} normals OK, {anom} anomalies leaked ({leak:.1f}%)")
    elif tier_val == 1:
        print(f"    {tier_label:>6s}: {n:>5d} ({pct:>5.1f}%)  |  {anom} anomalies + {norm} normals → human review")
    else:
        fa = norm / max(n, 1) * 100
        print(f"    {tier_label:>6s}: {n:>5d} ({pct:>5.1f}%)  |  {anom} real anomalies, {norm} false alarms ({fa:.1f}%)")

auto_rate = ((tiers == 0).sum() + (tiers == 2).sum()) / len(y_te) * 100
print(f"\n  Auto-decided: {auto_rate:.1f}%  |  Needs human: {(tiers == 1).sum()}")

  TIERED OUTPUT (Mammography, cost=10)
     CLEAR:  3141 ( 93.6%)  |  3128 normals OK, 13 anomalies leaked (16.7%)
    REVIEW:    54 (  1.6%)  |  9 anomalies + 45 normals → human review
     ALERT:   160 (  4.8%)  |  56 real anomalies, 104 false alarms (65.0%)

  Auto-decided: 98.4%  |  Needs human: 54


## Step 10: Shuttle Dataset — Best Configs
Repeat best ensemble configurations on shuttle (21.4% anomaly rate, easier but larger).

In [ ]:
# Cell 10: Load shuttle & run baseline + best ensembles
X_sh, y_sh = load_shuttle()

np.random.seed(42)
n_sub = 10_000
if len(X_sh) > n_sub:
    idx = np.random.choice(len(X_sh), n_sub, replace=False)
    X_sh, y_sh = X_sh[idx], y_sh[idx]

X_sh_tr, X_sh_te, y_sh_tr, y_sh_te = train_test_split(
    X_sh, y_sh, test_size=0.3, random_state=42, stratify=y_sh
)
print(f"Shuttle: train={len(X_sh_tr)}, test={len(X_sh_te)}, "
      f"anomaly rate={y_sh_te.mean():.1%}")

shuttle_results = {}

# Baseline QDA-Mag
p0 = UDLPipeline(projection_method='qda-magnified', cost_ratio=1.0)
p0.magnifier_.verbose = False
p0.fit(X_sh_tr, y_sh_tr)
shuttle_results['Baseline cost=1'] = evaluate("Shuttle Baseline c=1", p0, X_sh_te, y_sh_te)

p0c = UDLPipeline(projection_method='qda-magnified', cost_ratio=10.0)
p0c.magnifier_.verbose = False
p0c.fit(X_sh_tr, y_sh_tr)
shuttle_results['Baseline cost=10'] = evaluate("Shuttle Baseline c=10", p0c, X_sh_te, y_sh_te)

# Best ensembles
for mfls in ['quadratic', 'quadratic_smooth', 'logistic']:
    for mode in ['stack', 'product']:
        for cost in [1.0, 10.0]:
            cstr = f"c={int(cost)}"
            label = f"{mfls}+{mode} {cstr}"
            p = UDLPipeline(
                projection_method='qda-magnified',
                cost_ratio=cost,
                ensemble=mode,
                mfls_method=mfls,
            )
            p.magnifier_.verbose = False
            p.fit(X_sh_tr, y_sh_tr)
            shuttle_results[label] = evaluate(label, p, X_sh_te, y_sh_te)
            print()

print("\n=== SHUTTLE SUMMARY ===")
print(f"  {'Config':<35} {'Acc':>6} {'F1':>7} {'FP':>5} {'FN':>5} {'FP%':>6} {'FN%':>6}")
print(f"  {'-'*75}")
for name, r in shuttle_results.items():
    print(f"  {name:<35} {r['acc']:>5.1f}% {r['f1']:>7.4f} {r['fp']:>5} {r['fn']:>5} {r['fpr']:>5.2f}% {r['fnr']:>5.2f}%")

Shuttle: train=7000, test=3000, anomaly rate=21.4%


## Step 11: Final Comparison Table
Side-by-side comparison of ALL configurations across both datasets.

In [ ]:
# Cell 11: Final comparison table
print("=" * 90)
print("FINAL COMPARISON — MFLS + QDA-Magnified Ensemble Benchmark")
print("=" * 90)

for dataset_name, res_dict in [("MAMMOGRAPHY", results), ("SHUTTLE", shuttle_results)]:
    print(f"\n{'─' * 90}")
    print(f"  {dataset_name}")
    print(f"{'─' * 90}")
    print(f"  {'Configuration':<35} {'Acc':>6} {'F1':>7} {'FP':>5} {'FN':>5} {'FP%':>6} {'FN%':>6}")
    print(f"  {'-'*35} {'-'*6} {'-'*7} {'-'*5} {'-'*5} {'-'*6} {'-'*6}")
    for name, r in res_dict.items():
        print(f"  {name:<35} {r['acc']:>5.1f}% {r['f1']:>7.4f} {r['fp']:>5} {r['fn']:>5} {r['fpr']:>5.2f}% {r['fnr']:>5.2f}%")

# Identify best F1 for each dataset
print(f"\n{'=' * 90}")
print("BEST CONFIGURATIONS")
print(f"{'=' * 90}")
for dataset_name, res_dict in [("MAMMOGRAPHY", results), ("SHUTTLE", shuttle_results)]:
    best_name = max(res_dict, key=lambda k: res_dict[k]['f1'])
    best = res_dict[best_name]
    lowest_fn_name = min(res_dict, key=lambda k: res_dict[k]['fn'])
    lowest_fn = res_dict[lowest_fn_name]
    print(f"\n  {dataset_name}:")
    print(f"    Best F1:      {best_name} — F1={best['f1']:.4f}, FN={best['fn']}, FP={best['fp']}")
    print(f"    Lowest FN:    {lowest_fn_name} — FN={lowest_fn['fn']}, FP={lowest_fn['fp']}, F1={lowest_fn['f1']:.4f}")